In [2]:
pip install yfinance


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ---------------------------------------- 0.0/949.2 kB ? eta -:--:--
     -------------------- ----------------- 524.3/949.2 kB 4.9 MB/s eta 0:00:01
     ------------------------------- ------ 786.4/949.2 kB 6.2 MB/s eta 0:00:01
     -------------------------------------- 949.2/949.2 kB 2.0 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------------------- -------------------- 0.8/1.6 MB 3.0 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 4.0 MB/s eta 0:00:00
  Created wheel for multitasking:

  DEPRECATION: Building 'multitasking' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'multitasking'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [1]:
# --- Step 1: Install dependencies ---
# pip install yfinance pandas tqdm

import yfinance as yf
import pandas as pd
from tqdm import tqdm

# --- Step 2: List of 50 global + Indian companies (tickers) ---
tickers = [
    # US Companies
    "AAPL", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "NVDA", "JPM", "V", "PG",
    "JNJ", "XOM", "WMT", "DIS", "BAC", "NFLX", "PFE", "KO", "INTC", "PEP",
    
    # European Companies
    "SHEL.L", "HSBA.L", "BP.L", "ULVR.L", "RDSA.AS", "NESN.SW", "NOVN.SW", "SIE.DE", "BMW.DE", "VOW3.DE",
    
    # Indian Companies
    "RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS", "ICICIBANK.NS",
    "ITC.NS", "LT.NS", "SBIN.NS", "BHARTIARTL.NS", "BAJFINANCE.NS",
    
    # Asian & Others
    "SONY", "TM", "BABA", "TSM", "NVO", "RIO", "BHP", "SHOP", "TD", "RY"
]

# --- Step 3: Collect financial information ---
data_list = []

for ticker in tqdm(tickers):
    try:
        info = yf.Ticker(ticker).info
        data_list.append({
            "Company": info.get("longName"),
            "Ticker": ticker,
            "Country": info.get("country"),
            "Sector": info.get("sector"),
            "Revenue": info.get("totalRevenue"),
            "ProfitMargin": info.get("profitMargins"),
            "ROE": info.get("returnOnEquity"),
            "MarketCap": info.get("marketCap")
        })
    except Exception as e:
        print(f"Error fetching {ticker}: {e}")
        

# --- Step 4: Convert to DataFrame ---
df = pd.DataFrame(data_list)

# --- Step 5: Clean up data ---
df.dropna(subset=["Company"], inplace=True)
df.reset_index(drop=True, inplace=True)

print(df.head())


100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:38<00:00,  1.30it/s]

                 Company Ticker        Country                  Sector  \
0             Apple Inc.   AAPL  United States              Technology   
1  Microsoft Corporation   MSFT  United States              Technology   
2          Alphabet Inc.  GOOGL  United States  Communication Services   
3       Amazon.com, Inc.   AMZN  United States       Consumer Cyclical   
4   Meta Platforms, Inc.   META  United States  Communication Services   

        Revenue  ProfitMargin      ROE     MarketCap  
0  4.086250e+11       0.24296  1.49814  3.809380e+12  
1  2.817240e+11       0.36146  0.33281  3.928949e+12  
2  3.713990e+11       0.31118  0.34829  3.034565e+12  
3  6.700380e+11       0.10540  0.24770  2.355879e+12  
4  1.788040e+11       0.39992  0.40648  1.797839e+12  


### For revenue and marketcap, numbers are too large and difficult to read. So I will convert those columns in Billion so that it is easy to understand

In [2]:
df['Revenue_billion'] = (df['Revenue']/1e9).round(2)

In [3]:
df['MarketCap_billion'] = (df['MarketCap']/1e9).round(2)

### Drop revenue and marketcap column as I have new columns with values in billion

In [4]:
df.drop(columns = ['Revenue','MarketCap'], inplace=True)

In [5]:
# --- Save data to CSV ---
df.to_csv("companies.csv", index=False)

In [6]:
import pandas as pd
df = pd.read_csv("companies.csv")

In [7]:
df

,Company,Ticker,Country,Sector,ProfitMargin,ROE,Revenue_billion,MarketCap_billion
0,Apple Inc.,AAPL,United States,Technology,0.24296,1.49814,408.62,3809.38
1,Microsoft Corporation,MSFT,United States,Technology,0.36146,0.33281,281.72,3928.95
2,Alphabet Inc.,GOOGL,United States,Communication Services,0.31118,0.34829,371.40,3034.56
3,"Amazon.com, Inc.",AMZN,United States,Consumer Cyclical,0.10540,0.24770,670.04,2355.88
4,"Meta Platforms, Inc.",META,United States,Communication Services,0.39992,0.40648,178.80,1797.84
5,"Tesla, Inc.",TSLA,United States,Consumer Cyclical,0.06344,0.08177,92.72,1507.12
6,NVIDIA Corporation,NVDA,United States,Technology,0.52414,1.09417,165.22,4517.34
7,JPMorgan Chase & Co.,JPM,United States,Financial Services,0.34524,0.16211,163.75,850.17
8,Visa Inc.,V,United States,Financial Services,0.52158,0.51755,38.89,677.93
9,The Procter & Gamble Company,PG,United States,Consumer Defensive,0.18953,0.31242,84.28,352.03


### Handling missing values:

In [8]:
df.isnull().sum()

Company              0
Ticker               0
Country              0
Sector               0
ProfitMargin         0
ROE                  3
Revenue_billion      0
MarketCap_billion    0
dtype: int64

In [9]:
df[df['ROE'].isna()]

,Company,Ticker,Country,Sector,ProfitMargin,ROE,Revenue_billion,MarketCap_billion
29,Reliance Industries Limited,RELIANCE.NS,India,Energy,0.08346,NaN,9765.41,18705.94
34,ITC Limited,ITC.NS,India,Consumer Defensive,0.44154,NaN,790.40,5010.85
35,Larsen & Toubro Limited,LT.NS,India,Industrials,0.05903,NaN,2688.02,5134.49


### Handling missing values: 
### Fill missing ROE values with the median ROE of their respective sector.
### why:
#####   - ROE (Return on Equity) measures a company’s profitability relative to its shareholders' equity.
#####   - Missing ROE values can occur when financial data is incomplete or unavailable for a specific company.
#####   - Filling NaN with the sector median ensures:
#####      • Comparability across companies within the same industry.
#####       • Preservation of realistic values (instead of using 0, which may distort analysis).
#####      • Cleaner visualizations and calculations in Power BI and SQL.

In [10]:
df['ROE'] = df.groupby('Sector')['ROE'].transform(lambda x:x.fillna(x.median()))

In [11]:
df.isnull().sum()

Company              0
Ticker               0
Country              0
Sector               0
ProfitMargin         0
ROE                  0
Revenue_billion      0
MarketCap_billion    0
dtype: int64

In [12]:
df.describe()

,ProfitMargin,ROE,Revenue_billion,MarketCap_billion
count,49.000000,49.000000,49.000000,49.000000
mean,0.202692,0.270778,2001.676939,2604.581429
std,0.159179,0.269345,7176.975348,4230.457101
min,-0.386360,-0.186160,10.010000,46.890000
25%,0.092940,0.121680,63.440000,190.520000
50%,0.192150,0.200790,165.220000,434.160000
75%,0.311180,0.332810,693.150000,3809.380000
max,0.524140,1.498140,48452.150000,18705.940000


### Connection to SQL server

In [13]:
#connection to SQL server
import pyodbc
from sqlalchemy import create_engine
import urllib

server = 'ASIF-LAPTOP\SQLEXPRESS02'       
database = 'company'     
# Build connection string
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"  # or use UID/PWD if SQL Auth
)

# Create SQLAlchemy engine
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")


In [14]:
# List of dataframes and corresponding table names
table = {'company_details': df}


# Load each DataFrame to SQL
for table_name, df in table.items():
    df.to_sql(table_name, con=engine, if_exists='append', index=False)
    print(f"DataFrame loaded into SQL table: {table_name}")

DataFrame loaded into SQL table: company_details
